# WSJ 2027 - Push ledare-avdelningar till Scoutnet

Läser `input/Ledarlistan – till kommunikation.xlsx` (flik `Klara ledarteam med avdelning`),
matchar varje rad mot Scoutnet API:s deltagare för att hitta `member_no`,
och pushar `Deltagareavdelning` till question 88168 (Avdelning) via checkin-endpoint.

**Matchningsstrategi:**
1. **E-postmatch** (primär, hög konfidens) — `E-post` ↔ `primary_email`
2. **Namnmatch** (fallback) — `Förnamn Efternamn` ↔ `first_name + last_name` (endast om unik)
3. **MANUAL_OVERRIDES** — för rader som inte matchar (member_no eller `None` för att hoppa över)

**Output:** `output/wsj27_ledare_avdelningar.csv` (för manuell granskning innan push)

In [1]:
import sys, importlib
sys.path.insert(0, '/config/notebooks/wsj27')
import wsj27_utils as u
importlib.reload(u)

import pandas as pd

EXCEL_PATH = '/config/notebooks/wsj27/input/Ledarlistan – till kommunikation.xlsx'
SHEET = 'Klara ledarteam med avdelning'
OUTPUT_CSV = '/config/notebooks/wsj27/output/wsj27_ledare_avdelningar.csv'

# Manuella overrides för rader som inte matchas automatiskt.
# Nyckel = (förnamn, efternamn) exakt som i Excel.
# Värde = member_no (int) för att matcha mot specifik person,
#         None för att hoppa över (t.ex. inte registrerad i Scoutnet ännu).
MANUAL_OVERRIDES = {
    ('Johan', 'Wiklander'): 3003023,
    ('Joakim', 'Skogman'): 3429537,   # heter Joakim Huss i Scoutnet
}

In [2]:
# Läs Excel
df_xl = pd.read_excel(EXCEL_PATH, sheet_name=SHEET)
df_xl = df_xl.dropna(subset=['Förnamn', 'Efternamn'], how='all').copy()
df_xl['Förnamn'] = df_xl['Förnamn'].fillna('').astype(str).str.strip()
df_xl['Efternamn'] = df_xl['Efternamn'].fillna('').astype(str).str.strip()
df_xl['E-post'] = df_xl['E-post'].fillna('').astype(str).str.strip().str.lower()
df_xl['Deltagareavdelning'] = pd.to_numeric(df_xl['Deltagareavdelning'], errors='coerce').astype('Int64')

# Skala bort rader utan avdelning
missing_avd = df_xl[df_xl['Deltagareavdelning'].isna()]
if len(missing_avd):
    print(f'VARNING: {len(missing_avd)} rader saknar Deltagareavdelning - hoppas över:')
    for _, r in missing_avd.iterrows():
        print(f"  {r['Ledarteamsnr']}: {r['Förnamn']} {r['Efternamn']}")
    df_xl = df_xl.dropna(subset=['Deltagareavdelning']).copy()

print(f'\n{len(df_xl)} ledare-rader att processa')
print(f'Avdelningar: {sorted(df_xl["Deltagareavdelning"].unique().tolist())}')


211 ledare-rader att processa
Avdelningar: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53]


In [3]:
# Hämta Scoutnet och bygg lookup - BEGRÄNSAT till bekräftade avdelningsledare
import re, unicodedata

# Avdelningsledare-fee_ids
LEDARE_FEES = {'25695', '27560'}  # direktresa, rundresa

raw = u.fetch_participants()
parts = raw['participants']

def _norm(s):
    """Normalisera namn för fuzzy jämförelse: lowercase, strip accenter,
    strip (parenteser), bindestreck → mellanslag, kollapsa whitespace."""
    if not s:
        return ''
    s = unicodedata.normalize('NFKD', str(s).lower())
    s = ''.join(c for c in s if not unicodedata.combining(c))
    s = re.sub(r'\([^)]*\)', '', s)
    s = s.replace('-', ' ')
    return ' '.join(s.split())

def name_match(ef, el, sf, sl):
    """True om Excel-namn (ef, el) matchar Scoutnet-namn (sf, sl).
    Kräver: samma första-förnamn (efter normalisering) och minst ett
    gemensamt efternamns-ord. Hanterar accenter, mellannamn, bindestreck."""
    ef, el, sf, sl = _norm(ef), _norm(el), _norm(sf), _norm(sl)
    if not (ef and el and sf and sl):
        return False
    if ef.split()[0] != sf.split()[0]:
        return False
    return bool(set(el.split()) & set(sl.split()))

by_email = {}
sn_names = {}  # member_no -> (first_name, last_name)
skipped_cancelled = 0
skipped_unconfirmed = 0
for mno, p in parts.items():
    if str(p.get('fee_id', '')) not in LEDARE_FEES:
        continue
    if p.get('cancelled'):
        skipped_cancelled += 1
        continue
    if not p.get('confirmed'):
        skipped_unconfirmed += 1
        continue
    em = (p.get('primary_email') or '').strip().lower()
    if em:
        by_email.setdefault(em, []).append(int(mno))
    fn = (p.get('first_name') or '').strip()
    ln = (p.get('last_name') or '').strip()
    if fn and ln:
        sn_names[int(mno)] = (fn, ln)

print(f'Lookup (endast bekräftade avdelningsledare):')
print(f'  {len(sn_names)} ledare med namn')
print(f'  {len(by_email)} unika e-post')
print(f'  ({skipped_cancelled} cancelled, {skipped_unconfirmed} unconfirmed exkluderade)')

Fetched 2708 participants
Confirmed: 2621, Unconfirmed: 6, Cancelled: 81
Lookup (endast bekräftade avdelningsledare):
  209 ledare med namn
  209 unika e-post
  (1 cancelled, 3 unconfirmed exkluderade)


In [4]:
# Matcha varje ledare
matches = []
ambiguous = []
unmatched = []
skipped = []

for _, r in df_xl.iterrows():
    fn, ln = r['Förnamn'], r['Efternamn']
    email = r['E-post']
    avd = int(r['Deltagareavdelning'])
    team = r['Ledarteamsnr']
    base = {'team': team, 'avd': avd, 'first': fn, 'last': ln, 'email': email}

    # 1. Manual override
    key = (fn, ln)
    if key in MANUAL_OVERRIDES:
        mno = MANUAL_OVERRIDES[key]
        if mno is None:
            skipped.append({**base, 'reason': 'manual=None'})
        else:
            matches.append({**base, 'member_no': int(mno), 'match_by': 'manual'})
        continue

    # 2. Email-match — kräv namn-match också, annars fall through
    email_outcome = None
    if email and email in by_email:
        mnos = by_email[email]
        name_ok = [m for m in mnos if m in sn_names and name_match(fn, ln, *sn_names[m])]
        if len(name_ok) == 1:
            label = 'email' if len(mnos) == 1 else 'email+name'
            matches.append({**base, 'member_no': name_ok[0], 'match_by': label})
            continue
        if len(name_ok) > 1:
            cands = [(m, f'{sn_names[m][0]} {sn_names[m][1]}') for m in name_ok]
            ambiguous.append({**base, 'reason': f'email {email} -> flera namnpassande: {cands}'})
            continue
        # Email matchar men namnet stämmer inte med någon - spara för fall-through
        cands = [(m, f'{sn_names[m][0]} {sn_names[m][1]}' if m in sn_names else '?') for m in mnos]
        email_outcome = f'email {email} -> {cands} (namn passar inte)'

    # 3. Fuzzy namn-match (linjär sökning över alla Scoutnet)
    cands = [m for m, (sf, sl) in sn_names.items() if name_match(fn, ln, sf, sl)]
    if len(cands) == 1:
        matches.append({**base, 'member_no': cands[0], 'match_by': 'name'})
        continue
    if len(cands) > 1:
        detail = [(m, f'{sn_names[m][0]} {sn_names[m][1]}') for m in cands]
        ambiguous.append({**base, 'reason': f'name {fn} {ln} -> {detail}'})
        continue

    if email_outcome:
        ambiguous.append({**base, 'reason': email_outcome})
    else:
        unmatched.append(base)

print(f'=== Resultat ===')
by_kind = {}
for m in matches:
    by_kind[m['match_by']] = by_kind.get(m['match_by'], 0) + 1
print(f'  Matchade:  {len(matches)} ({by_kind})')
print(f'  Tvetydiga: {len(ambiguous)}')
print(f'  Omatchade: {len(unmatched)}')
print(f'  Hoppade över (manual=None): {len(skipped)}')

if ambiguous:
    print('\n--- Tvetydiga (behöver MANUAL_OVERRIDES) ---')
    for a in ambiguous:
        print(f"  {a['team']:>4} avd {a['avd']:>2}: {a['first']} {a['last']} ({a['email']}) -> {a['reason']}")

if unmatched:
    print('\n--- Omatchade (behöver MANUAL_OVERRIDES) ---')
    for u_ in unmatched:
        print(f"  {u_['team']:>4} avd {u_['avd']:>2}: {u_['first']} {u_['last']} ({u_['email']})")

=== Resultat ===
  Matchade:  208 ({'email': 155, 'name': 51, 'manual': 2})
  Tvetydiga: 0
  Omatchade: 3
  Hoppade över (manual=None): 0

--- Omatchade (behöver MANUAL_OVERRIDES) ---
   D09 avd 15: Hannah Narkun (hannah.narkun@ornsberg.org)
   D10 avd 16: Gustaf Runius (gustaf.s.runius@gmail.com)
   D12 avd 12: Hanna Dalemark (hannadalemark@hotmail.com)


In [5]:
# Sammanställning per avdelning - 4 ledare förväntas, flagga avvikelser
EXPECTED_PER_AVD = 4

# Bygg per-avd-status från alla kategorier
status_by_avd = {}  # avd -> {'matched': [...], 'problem': [...], 'team': str}
for m in matches:
    s = status_by_avd.setdefault(int(m['avd']), {'matched': [], 'problem': [], 'team': m['team']})
    s['matched'].append(f"{m['first']} {m['last']} ({m['match_by']})")
for a in ambiguous + unmatched + skipped:
    s = status_by_avd.setdefault(int(a['avd']), {'matched': [], 'problem': [], 'team': a['team']})
    label = 'TVETYDIG' if a in ambiguous else ('OMATCHAD' if a in unmatched else 'SKIPPAD')
    s['problem'].append(f"{a['first']} {a['last']} [{label}]")

print(f'{"Avd":>3} {"Team":>4} {"#":>2}/{EXPECTED_PER_AVD}  Status')
print('-' * 80)
ok_count = wrong_count_count = problem_count = 0
for avd in sorted(status_by_avd.keys()):
    s = status_by_avd[avd]
    n_ok = len(s['matched'])
    n_prob = len(s['problem'])
    total = n_ok + n_prob
    has_problem = n_prob > 0
    diff = total - EXPECTED_PER_AVD

    if has_problem:
        status = 'AVVIKELSE'; problem_count += 1
    elif diff > 0:
        status = f'FÖR MÅNGA ({diff:+d})'; wrong_count_count += 1
    elif diff < 0:
        status = f'FÖR FÅ ({diff:+d})'; wrong_count_count += 1
    else:
        status = 'OK'; ok_count += 1

    print(f'{avd:>3} {s["team"]:>4} {n_ok:>2}/{EXPECTED_PER_AVD}  {status}')
    if has_problem:
        for p in s['problem']:
            print(f'         - {p}')

print(f'\n=== {ok_count} OK | {wrong_count_count} fel antal i Excel | {problem_count} med matchningsavvikelser ===')

Avd Team  #/4  Status
--------------------------------------------------------------------------------
  1  D03  4/4  OK
  2  D02  4/4  OK
  3  D05  5/4  FÖR MÅNGA (+1)
  4  D06  4/4  OK
  5  D04  4/4  OK
  6  D13  4/4  OK
  7  D16  4/4  OK
  8  D15  4/4  OK
  9  D01  4/4  OK
 10  D08  4/4  OK
 11  D14  4/4  OK
 12  D12  3/4  AVVIKELSE
         - Hanna Dalemark [OMATCHAD]
 13  D11  4/4  OK
 14  D07  3/4  FÖR FÅ (-1)
 15  D09  3/4  AVVIKELSE
         - Hannah Narkun [OMATCHAD]
 16  D10  3/4  AVVIKELSE
         - Gustaf Runius [OMATCHAD]
 17  R19  4/4  OK
 18  R20  4/4  OK
 19  R30  4/4  OK
 20  R23  4/4  OK
 21  R24  4/4  OK
 22  R22  4/4  OK
 23  R27  4/4  OK
 24  R37  4/4  OK
 25  R35  4/4  OK
 26  R33  4/4  OK
 27  R34  4/4  OK
 28  R32  4/4  OK
 29  R31  4/4  OK
 30  R21  4/4  OK
 31  R07  4/4  OK
 32  R15  4/4  OK
 33  R06  4/4  OK
 34  R13  3/4  FÖR FÅ (-1)
 35  R14  4/4  OK
 36  R36  4/4  OK
 37  R26  4/4  OK
 38  R25  4/4  OK
 39  R29  4/4  OK
 40  R10  4/4  OK
 41  R11  4/4  OK

In [6]:
# Kontrollera om det finns bekräftade avdelningsledare i Scoutnet som saknas i Excel
LEDARE_FEE_LABEL = {'25695': 'direktresa', '27560': 'rundresa'}

scoutnet_ledare = {}  # member_no -> (first, last, email, fee_label, kar)
for mno_str, p in parts.items():
    fee = str(p.get('fee_id', ''))
    if fee not in LEDARE_FEE_LABEL:
        continue
    if p.get('cancelled') or not p.get('confirmed'):
        continue
    pmi = p.get('primary_membership_info')
    kar = ''
    if isinstance(pmi, dict):
        kar = pmi.get('group_name') or ''
    scoutnet_ledare[int(mno_str)] = (
        (p.get('first_name') or '').strip(),
        (p.get('last_name') or '').strip(),
        (p.get('primary_email') or '').strip(),
        LEDARE_FEE_LABEL[fee],
        kar,
    )

print(f'Totalt bekräftade avdelningsledare i Scoutnet: {len(scoutnet_ledare)}')
print(f'Matchade från Excel: {len(matches)}')

excel_member_nos = {int(m['member_no']) for m in matches}
saknas_i_excel = set(scoutnet_ledare.keys()) - excel_member_nos

print(f'\nLedare i Scoutnet som INTE finns i Excel: {len(saknas_i_excel)}')
if saknas_i_excel:
    print(f'{"Mno":>8} {"Resetyp":>10}  Namn (e-post) - kår')
    for mno in sorted(saknas_i_excel):
        fn, ln, em, fee_label, kar = scoutnet_ledare[mno]
        print(f'{mno:>8} {fee_label:>10}  {fn} {ln} ({em}) - {kar}')

Totalt bekräftade avdelningsledare i Scoutnet: 209
Matchade från Excel: 208

Ledare i Scoutnet som INTE finns i Excel: 1
     Mno    Resetyp  Namn (e-post) - kår
 3287453 direktresa  Victor Tollbom (victortollbom@gmail.com) - Johannebergs Scoutkår


In [7]:
# Bygg DataFrame och spara CSV för manuell granskning
import os
df_out = pd.DataFrame(matches).sort_values(['avd', 'last', 'first']).reset_index(drop=True)
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
df_out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'Sparade {len(df_out)} rader till {OUTPUT_CSV}')

# Sanity-check: dubbletter på member_no (samma person på flera ledarteam)
dups = df_out[df_out.duplicated('member_no', keep=False)]
if len(dups):
    print(f'\nVARNING: {len(dups)} rader med dubbla member_no (samma person på flera avdelningar):')
    for mno, grp in dups.groupby('member_no'):
        teams = ', '.join(f"{r['team']}=avd{r['avd']}" for _, r in grp.iterrows())
        print(f"  {mno} {grp.iloc[0]['first']} {grp.iloc[0]['last']}: {teams}")

df_out.head(10)

Sparade 208 rader till /config/notebooks/wsj27/output/wsj27_ledare_avdelningar.csv


,team,avd,first,last,email,member_no,match_by
0,D03,1,Johanna,Gersne,johannagersne@gmail.com,3247255,email
1,D03,1,Julia,Hemmendy,julia.hemmendy@gmail.com,3211155,email
2,D03,1,Joakim,Munther Fd Pålsson,joakim@palssongard.se,3245396,email
3,D03,1,Jonas,Svanlund,jonas.svanlund@gmail.com,3008254,email
4,D02,2,Sara,Bengtsson,sara.a.bengtsson@gmail.com,3175993,email
5,D02,2,Alice,Hemmendy,alice299@hotmail.com,3216071,email
6,D02,2,Olof,Olsson,olofo02@gmail.com,3200622,email
7,D02,2,Emil,Vikström,rsemil@gmail.com,3243151,email
8,D05,3,Alexander,Blomqvist,alexanderblomqvist94@gmail.com,3149703,email
9,D05,3,Ann-Louise,Ekestubbe,a.ekestubbe@gmail.com,3383095,email


In [11]:
# DRY RUN - för LEDARE använder vi Q107592 (form 47115 Avdelningsledare),
# inte Q88168 som är form 39188 (Deltagare/IST).
LEDARE_AVDELNING_QID = '107592'

member_to_avd = dict(zip(df_out['member_no'].astype(int), df_out['avd'].astype(int)))
u.push_avdelningar_to_scoutnet(member_to_avd, dry_run=True, qid=LEDARE_AVDELNING_QID)

Total att skicka: 208 (hoppade över 0)

DRY RUN - inget skickas. Första 3 i payload:
  3247255: {'questions': {'107592': {'value': '1'}}}
  3211155: {'questions': {'107592': {'value': '1'}}}
  3245396: {'questions': {'107592': {'value': '1'}}}


{'updated': 0,
 'unchanged': 0,
 'not_found': 0,
 'no_member': 0,
 'skipped': 0,
 'would_send': 208,
 'dry_run': True}

In [12]:
# SKARP PUSH till Q107592 - avkommentera när dry-run ser rätt ut
u.push_avdelningar_to_scoutnet(member_to_avd, dry_run=False, qid=LEDARE_AVDELNING_QID)

Total att skicka: 208 (hoppade över 0)
  Chunk 1: updated=1 unchanged=199 not_found=0 no_member=0
  Chunk 2: updated=2 unchanged=206 not_found=0 no_member=0

=== Totalt ===
  Uppdaterade (faktiskt ändrade): 2
  Oförändrade (redan rätt värde): 206
  Hittades inte:                  0
  Saknar medlemskap:              0
  Överhoppade (Avd 99 mm):        0


{'updated': 2,
 'unchanged': 206,
 'not_found': 0,
 'no_member': 0,
 'skipped': 0,
 'errors': []}